# 🌍 Quick Start — Your First Jiskta Query

Welcome to the **Jiskta Climate Data API**.  
In this notebook you will install the SDK, connect with your API key, and plot daily NO₂ concentrations over Paris for a full month.

**Requirements**
```
pip install jiskta matplotlib
```

In [ ]:
# Install the SDK from the private GitHub repo.
# !pip install "jiskta[pandas]" matplotlib


## 1 — Connect

In [ ]:
import os
from jiskta import JisktaClient

API_KEY = os.environ.get("JISKTA_API_KEY", "sk_live_YOUR_API_KEY")  # or paste your key here

client = JisktaClient(api_key=API_KEY)
print("Client ready ✓")

## 2 — Query daily NO₂ over Paris (January 2023)

The bounding box covers the Paris metropolitan area (≈ 0.3° × 0.3°, 9 grid points).

In [ ]:
df = client.query(
    lat=(48.7, 49.0),
    lon=(2.2, 2.5),
    start="2023-01",
    end="2023-01",
    variables=["no2"],
    aggregate="daily",
)

print(f"Rows: {len(df)}")
print(f"Credits used in this query: included in response metadata")
df.head(10)

## 3 — Plot

We area-average the grid points so we get one value per day.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Average over all grid points for a single time series
ts = df.groupby("date")["no2_mean"].mean().reset_index()
ts["date"] = ts["date"].astype("datetime64[ns]")

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(ts["date"], ts["no2_mean"], alpha=0.18, color="#4c9be8")
ax.plot(ts["date"], ts["no2_mean"], color="#1a6bbf", linewidth=2, marker="o", markersize=4)

# WHO daily guideline for NO₂ = 25 µg/m³
ax.axhline(25, color="#e05252", linewidth=1.2, linestyle="--", label="WHO guideline (25 µg/m³)")

ax.set_title("Daily NO₂ — Paris metropolitan area, January 2023", fontsize=14, pad=12)
ax.set_ylabel("NO₂  (µg/m³)")
ax.set_xlabel("")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 4 — Summary statistics (cheapest format)

`client.stats()` uses `format=stats` — no CSV output, minimal credits.

In [ ]:
result = client.stats(
    lat=(48.7, 49.0),
    lon=(2.2, 2.5),
    start="2023-01",
    end="2023-01",
    variables=["no2"],
)
print(result["output"])
print(f"\nCredits remaining: {result['credits_remaining']}")

---
**Next:** See [02_air_quality_report.ipynb](02_air_quality_report.ipynb) for a full-year multi-pollutant analysis.